# Confusion Pairs for Musicological Annotation

**Purpose:** identify the cantaor pairs whose embeddings are most similar (lowest cosine
distance), ranked for manual annotation. For each pair of distinct cantaores, the
*minimum* cosine distance across all combinations of their recordings is reported —
this is the best-case confusion between those two singers in the embedding space.

Block assignment follows a two-level rule. **within-TONAS** is cantaor-level; the
remaining blocks and sub-blocks are determined by the sources of the recordings that
achieve the minimum distance:

| Block | Sub-block | Criterion | Interpretive status |
|---|---|---|---|
| **clean** | — | Both recordings cante100 | Same acoustic domain, varied palos — cleanest comparison |
| **domain-flagged** | cross-domain pair | Recordings from different sources (one TONAS, one cante100) | Similarity may reflect acoustic-domain proximity in either direction (source probe 0.979) |
| **domain-flagged** | cross-dataset cantaor, same-domain match | Both recordings TONAS, but ≥1 cantaor also has cante100 recordings | Closeness likely inflated by TONAS domain clustering; the model has seen that cantaor in both conditions — not evidence of cross-dataset generalization |
| **within-TONAS** | — | Both cantaores exclusively in TONAS (C(6,2) = 15 pairs) | Same a cappella archival domain; similarity may reflect vocal style, recording format, or palo-family (tonás/martinete) — cannot be disentangled |

**`n_recordings_A/B`**: total recordings of each cantaor across the full 109-recording
dataset. Cantaores with many recordings have more chances to land a low minimum distance
by combinatorial effect — use this column to weight confidence in the ranking.

**No musicological interpretation is added here.** Annotation is performed externally.

In [ ]:
import os
import numpy as np
import pandas as pd

# Notebooks use relative paths — run from the project root (cante-embeddings/).
if not os.path.exists('data/processed'):
    raise RuntimeError("Run this notebook from the project root (cante-embeddings/).")

## 1. Load embeddings and reconstruct metadata

In [2]:
results      = np.load('data/processed/final_results.npz', allow_pickle=True)
embeddings   = results['embeddings']      # (109, 128)
recording_id = results['recording_id']
norm_name    = results['norm_name']

# --- source ---
split_df = pd.read_csv('data/processed/train_test_split.csv')

# --- palo: cante100 ---
c100_cat = pd.read_csv(
    'data/raw/cante100_catalogo - cante100_catalogo.csv.csv'
)[['filename', 'palo_normalizado']].rename(
    columns={'filename': 'recording_id', 'palo_normalizado': 'palo'}
)

# --- palo: TONAS (simplify sub-group labels to Debla / Martinete) ---
tonas_cat = pd.read_csv(
    'data/raw/tonas_catalogo - tonas_catalogo.csv.csv'
)[['archivo_wav', 'palo']].rename(columns={'archivo_wav': 'recording_id'})
tonas_cat['palo'] = tonas_cat['palo'].str.replace(
    r'Martinete \(grupo \d\)', 'Martinete', regex=True
)

palo_map = pd.concat([c100_cat, tonas_cat], ignore_index=True)

# Build per-recording DataFrame
recs = pd.DataFrame({
    'emb_idx':      range(len(recording_id)),
    'recording_id': recording_id,
    'norm_name':    norm_name,
})
recs = recs.merge(split_df[['recording_id', 'source']], on='recording_id', how='left')
recs = recs.merge(palo_map, on='recording_id', how='left')
recs.loc[recs['recording_id'] == '092_AntonioMairenaTonas.mp3', 'palo'] = 'Tonás'

# --- display names ---
nombre_map = pd.read_csv('data/processed/conteo_cantaores_combinado.csv', index_col=0)[
    'nombre'
].to_dict()
# rafaelromero_merged → rafaelromero in recs; map both
if 'rafaelromero_merged' in nombre_map and 'rafaelromero' not in nombre_map:
    nombre_map['rafaelromero'] = nombre_map['rafaelromero_merged']
recs['display_name'] = recs['norm_name'].map(nombre_map).fillna(recs['norm_name'])

print(f"Recordings loaded: {len(recs)}")
print(f"Cantaores: {recs['norm_name'].nunique()}")
print(f"Missing palo: {recs['palo'].isna().sum()}")

Recordings loaded: 109
Cantaores: 31
Missing palo: 0


## 2. Build per-cantaor metadata: all palos and sources

For each cantaor we collect:
- All unique palos across their recordings in the 109-recording dataset
- All sources (tonas / cante100) they appear in

In [3]:
cantaor_meta = (
    recs.groupby('norm_name')
    .agg(
        display_name=('display_name', 'first'),
        palos=('palo',   lambda x: ', '.join(sorted(x.dropna().unique()))),
        sources=('source', lambda x: ', '.join(sorted(x.unique()))),
        n_recs=('recording_id', 'count'),
    )
    .reset_index()
)

print(f"Cantaores: {len(cantaor_meta)}")
cantaor_meta.head(8)

Cantaores: 31


,norm_name,display_name,palos,sources,n_recs
0,antoniomairena,Antonio Mairena,"Alegrías, Bulerías, Debla, Malagueñas-Granaína...","cante100, tonas",14
1,antonionunezchocolate,"Antonio Nunez ""Chocolate""","Debla, Martinete",tonas,4
2,antoniopinana,Antonio Piñana,Cantes Mineros,cante100,2
3,aurelioselles,Aurelio Sellés,"Seguiriyas, Tientos-Tangos",cante100,2
4,bernardoloslobitos,"Bernardo ""El de los Lobitos""",Malagueñas-Granaínas,cante100,2
5,camaron,Camarón,"Alegrías, Cantes Mineros, Cantes de Ida y Vuel...",cante100,5
6,chanolobato,Chano Lobato,"Debla, Martinete",tonas,2
7,chaqueta,El Chaqueta,Martinete,tonas,2


## 3. Compute minimum pairwise distance for every cantaor pair

For each of the C(31, 2) = 465 pairs of distinct cantaores, we find the recording
combination that achieves the lowest cosine distance.

Block and sub-block assignment:
- **within-TONAS** (cantaor-level): both cantaores exclusively in TONAS → exactly 15 pairs.
- **clean** (recording-level): both recordings cante100.
- **domain-flagged / cross-domain pair**: recordings from different sources (tonas vs cante100).
- **domain-flagged / cross-dataset cantaor, same-domain match**: both recordings TONAS, but
  at least one cantaor also has cante100 recordings (cross-dataset cantaor).

In [4]:
cantaores = cantaor_meta['norm_name'].tolist()
n = len(cantaores)

# The 6 cantaores that appear exclusively in TONAS (no cante100 recordings)
tonas_only_set = set(cantaor_meta.loc[cantaor_meta['sources'] == 'tonas', 'norm_name'])
print(f"TONAS-exclusive cantaores ({len(tonas_only_set)}): {sorted(tonas_only_set)}")
print(f"Expected within-TONAS pairs: C({len(tonas_only_set)},2) = {len(tonas_only_set)*(len(tonas_only_set)-1)//2}")

# Fast lookups
by_cantaor  = {c: recs[recs['norm_name'] == c].reset_index(drop=True) for c in cantaores}
n_recs_map  = cantaor_meta.set_index('norm_name')['n_recs'].to_dict()
display_map = cantaor_meta.set_index('norm_name')['display_name'].to_dict()
palos_map   = cantaor_meta.set_index('norm_name')['palos'].to_dict()

rows = []
for i in range(n):
    for j in range(i + 1, n):
        ca, cb = cantaores[i], cantaores[j]
        recs_a, recs_b = by_cantaor[ca], by_cantaor[cb]

        best_dist = np.inf
        best_ra = best_rb = None
        for _, ra in recs_a.iterrows():
            ea = embeddings[int(ra['emb_idx'])]
            for _, rb in recs_b.iterrows():
                eb = embeddings[int(rb['emb_idx'])]
                d  = 1 - float(np.dot(ea, eb))
                if d < best_dist:
                    best_dist = d
                    best_ra, best_rb = ra, rb

        sa, sb = best_ra['source'], best_rb['source']

        if ca in tonas_only_set and cb in tonas_only_set:
            block     = 'within-TONAS'
            block_sub = ''
        elif sa == 'cante100' and sb == 'cante100':
            block     = 'clean'
            block_sub = ''
        else:
            block = 'domain-flagged'
            block_sub = 'cross-domain pair' if sa != sb else 'cross-dataset cantaor, same-domain match'

        rows.append({
            'cantaor_A':      display_map[ca],
            'cantaor_B':      display_map[cb],
            'n_recordings_A': n_recs_map[ca],
            'n_recordings_B': n_recs_map[cb],
            'palos_A':        palos_map[ca],
            'palos_B':        palos_map[cb],
            'source_A':       sa,
            'source_B':       sb,
            'recording_A':    best_ra['recording_id'],
            'recording_B':    best_rb['recording_id'],
            'cosine_dist':    round(best_dist, 4),
            'block':          block,
            'block_sub':      block_sub,
        })

all_pairs = pd.DataFrame(rows)
print(f"\nTotal cantaor pairs: {len(all_pairs)}")
print(f"Block counts:\n{all_pairs['block'].value_counts().to_string()}")
print(f"\nDomain-flagged sub-types:")
print(all_pairs[all_pairs['block']=='domain-flagged']['block_sub'].value_counts().to_string())

TONAS-exclusive cantaores (6): ['antonionunezchocolate', 'chanolobato', 'chaqueta', 'enriquemorente', 'naranjitotriana', 'pedrosanz']
Expected within-TONAS pairs: C(6,2) = 15

Total cantaor pairs: 465
Block counts:
block
domain-flagged    228
clean             222
within-TONAS       15

Domain-flagged sub-types:
block_sub
cross-domain pair                           136
cross-dataset cantaor, same-domain match     92


## 4. Display — by block and sub-block, sorted by cosine distance

For `domain-flagged`, a diagnostic shows the top-5 genuinely cross-domain pairs
(src_A ≠ src_B) before the full sub-block tables, as a finding in its own right.

In [5]:
TOP_N = 20

WARNINGS = {
    'clean': '',
    'cross-domain pair':
        'WARNING: recordings from different sources (one TONAS, one cante100). '
        'Similarity may reflect acoustic-domain proximity in either direction '
        '(source probe balanced accuracy 0.979).',
    'cross-dataset cantaor, same-domain match':
        'WARNING: both recordings are from TONAS, but ≥1 cantaor also has cante100 '
        'recordings. Closeness likely inflated by TONAS domain clustering — '
        'not evidence of cross-dataset vocal generalization.',
    'within-TONAS':
        'NOTE: both cantaores exclusively in TONAS (a cappella archival). Similarity '
        'may reflect genuine vocal style, shared a cappella recording format, or '
        'palo-family similarity (all cantes are tonás/martinete) — '
        'cannot be disentangled with this data alone.',
}

DCOLS = ['cantaor_A', 'n_recordings_A', 'cantaor_B', 'n_recordings_B',
         'palos_A', 'palos_B', 'source_A', 'source_B', 'cosine_dist']

top_frames = []

# ── CLEAN ──────────────────────────────────────────────────────────────────
clean_all = all_pairs[all_pairs['block'] == 'clean'].sort_values('cosine_dist')
sub = clean_all.head(TOP_N).copy()
sub['warning'] = WARNINGS['clean']
top_frames.append(sub)
print(f"{'='*72}")
print(f"BLOCK: CLEAN  ({len(sub)} shown of {len(clean_all)} total)")
print('='*72)
print(sub[DCOLS].to_string(index=False))

# ── DOMAIN-FLAGGED ─────────────────────────────────────────────────────────
df_all = all_pairs[all_pairs['block'] == 'domain-flagged']
cross_domain = df_all[df_all['block_sub'] == 'cross-domain pair'].sort_values('cosine_dist')
same_dom     = df_all[df_all['block_sub'] == 'cross-dataset cantaor, same-domain match'].sort_values('cosine_dist')

print(f"\n{'='*72}")
print(f"BLOCK: DOMAIN-FLAGGED  ({len(df_all)} total)")
print(f"  sub-type 'cross-domain pair'                       : {len(cross_domain)} pairs")
print(f"  sub-type 'cross-dataset cantaor, same-domain match': {len(same_dom)} pairs")
print('='*72)

# Diagnostic: top-5 genuinely cross-domain pairs
print(f"\n  ── FINDING: top-5 genuinely cross-domain pairs (src_A ≠ src_B) ──")
if len(cross_domain) == 0:
    print("  No genuinely cross-domain pairs (tonas/cante100) exist in this block.")
    print("  All domain-flagged pairs achieve their minimum within the same source.")
else:
    print(cross_domain.head(5)[DCOLS].to_string(index=False))

# Sub-block: cross-domain pair
print(f"\n  Sub-block: 'cross-domain pair'  ({len(cross_domain)} total, top-{min(TOP_N,len(cross_domain))} shown)")
print(f"  {WARNINGS['cross-domain pair']}")
sub_cd = cross_domain.head(TOP_N).copy()
sub_cd['warning'] = WARNINGS['cross-domain pair']
top_frames.append(sub_cd)
if len(sub_cd) > 0:
    print(sub_cd[DCOLS].to_string(index=False))

# Sub-block: cross-dataset cantaor, same-domain match
print(f"\n  Sub-block: 'cross-dataset cantaor, same-domain match'  ({len(same_dom)} total, top-{min(TOP_N,len(same_dom))} shown)")
print(f"  {WARNINGS['cross-dataset cantaor, same-domain match']}")
sub_sd = same_dom.head(TOP_N).copy()
sub_sd['warning'] = WARNINGS['cross-dataset cantaor, same-domain match']
top_frames.append(sub_sd)
if len(sub_sd) > 0:
    print(sub_sd[DCOLS].to_string(index=False))

# ── WITHIN-TONAS ───────────────────────────────────────────────────────────
wt = all_pairs[all_pairs['block'] == 'within-TONAS'].sort_values('cosine_dist').copy()
wt['warning'] = WARNINGS['within-TONAS']
top_frames.append(wt)
print(f"\n{'='*72}")
print(f"BLOCK: WITHIN-TONAS  ({len(wt)} of {len(wt)} total — all shown)")
print(f"  {WARNINGS['within-TONAS']}")
print('='*72)
print(wt[DCOLS].to_string(index=False))

output = pd.concat(top_frames, ignore_index=True)

BLOCK: CLEAN  (20 shown of 222 total)
                   cantaor_A  n_recordings_A                    cantaor_B  n_recordings_B                                                                                     palos_A                                                                   palos_B source_A source_B  cosine_dist
                 Juan Talega               6      Tía Anica "La Piriñaca"               3                                                             Martinete, Seguiriyas, Soleares                                           Martinete, Seguiriyas, Soleares cante100 cante100       0.0692
             Antonio Mairena              14        Manuel Soto "Sordera"               7 Alegrías, Bulerías, Debla, Malagueñas-Granaínas, Martinete, Soleares, Tientos-Tangos, Tonás                             Alegrías, Fandangos, Soleares, Tientos-Tangos cante100 cante100       0.0881
                 Juan Talega               6       Luis Torres "Joselero"               2            

## 5. Export

Full table saved to `data/processed/confusion_pairs_for_annotation.csv`.
Columns include recording filenames for traceability.

In [6]:
col_order = [
    'block', 'block_sub',
    'cantaor_A', 'cantaor_B',
    'n_recordings_A', 'n_recordings_B',
    'palos_A', 'palos_B',
    'source_A', 'source_B',
    'recording_A', 'recording_B',
    'cosine_dist', 'warning',
]
output[col_order].to_csv(
    'data/processed/confusion_pairs_for_annotation.csv',
    index=False
)
print(f"Saved: data/processed/confusion_pairs_for_annotation.csv")
print(f"Rows: {len(output)}  |  Columns: {len(col_order)}")
print(f"\nBlock / sub-block summary:")
print(output.groupby(['block', 'block_sub']).size().rename('n_pairs').to_string())

Saved: data/processed/confusion_pairs_for_annotation.csv
Rows: 75  |  Columns: 14

Block / sub-block summary:
block           block_sub                               
clean                                                       20
domain-flagged  cross-dataset cantaor, same-domain match    20
                cross-domain pair                           20
within-TONAS                                                15
